# 半监督 GNN 空间转录组细胞类型注释
## 转录本级别流水线（不依赖细胞分割）

**数据集**：GSE264393（健康肾脏 snRNA-seq）+ GSE264334（IgAN 肾脏 Xenium）

**方法对比**：
| | TopACT | 本方法（GNN）|
|---|---|---|
| 聚合方式 | 固定半径（2-5μm），人工设定 | 消息传递，端到端学习 |
| 分类器 | SVM（各尺度独立） | GCN/GraphSAGE/GAT |
| 空间信息 | 输入端聚合 | 图结构 + 空间 kNN 边 |
| 跨模态对齐 | 无 | MMD 域适应损失 |

In [ ]:
# ============================================================
# Cell 1: 环境配置
# ============================================================
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

PATHS = {
    'raw': {
        'transcripts': './data/raw/xenium/GSE264334_RAW/transcripts.csv.gz',
        'cells_csv'  : './data/raw/xenium/GSE264334_RAW/cells.csv.gz',
    },
    'cache': {
        'scrna'  : './data/cache/kidney_scrna_processed.rds',
        'xenium' : './data/cache/kidney_xenium_processed.rds',
        'graph'  : './data/cache/graph_spot/',
    },
    'output': {
        'models'     : './results/models/',
        'predictions': './results/predictions/',
        'plots'      : './plots/',
    },
}
for d in [*PATHS['output'].values(), PATHS['cache']['graph']]:
    os.makedirs(d, exist_ok=True)

PARAMS = {
    # 转录本 → spot
    'bin_size'       : 5,    # μm
    'qv_threshold'   : 20,
    'min_transcripts': 1,
    # 图构建
    'k_feat'         : 15,
    'k_spatial'      : 10,
    'val_ratio'      : 0.2,
    # 模型
    'hidden_dim'     : 256,
    'proj_dim'       : 512,
    'dropout'        : 0.5,
    # 训练
    'n_epochs'       : 500,
    'lr'             : 1e-3,
    'weight_decay'   : 5e-4,
    'patience'       : 40,
    'lr_factor'      : 0.5,
    'lr_patience'    : 15,
    'min_lr'         : 1e-5,
    'max_grad_norm'  : 1.0,
    'warmup_epochs'  : 30,
    # 半监督损失权重
    'lambda_mmd'     : 0.1,
    'lambda_ent'     : 0.01,
    'lambda_pl'      : 0.3,
    'pl_threshold'   : 0.90,
    'pl_update_freq' : 10,
    'device'         : 'cpu',
    'seed'           : 42,
}

from utils import set_seed
set_seed(PARAMS['seed'])
from gpu_utils import list_gpus, select_gpu
print('GPU 状态：')
list_gpus(show_processes=True)
PARAMS['device'] = select_gpu('auto', min_free_gb=8.0)
print(f'\n训练设备: {PARAMS["device"]}')

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R -o flex_expr -o flex_labels -o cell_types -o xenium_panel_genes
# ============================================================
# Cell 2: 从 labelTransfer_kidney.ipynb 缓存导出 scRNA 数据
# ============================================================
library(Seurat)

cat("Loading kidney scRNA reference...\n")
scrna_obj  <- readRDS("./data/cache/kidney_scrna_processed.rds")
xenium_obj <- readRDS("./data/cache/kidney_xenium_processed.rds")

# Xenium panel 基因列表（用于特征对齐）
xenium_panel_genes <- rownames(xenium_obj)
common_genes       <- intersect(rownames(scrna_obj), xenium_panel_genes)
cat("Xenium panel:", length(xenium_panel_genes),
    "| scRNA 基因:", nrow(scrna_obj),
    "| 共同基因:", length(common_genes), "\n")

# 导出 scRNA raw counts（只取共同基因，行=细胞, 列=基因）
flex_counts <- as.matrix(
    GetAssayData(scrna_obj, layer = "counts")[common_genes, ]
)
flex_expr <- t(flex_counts)   # (n_cells, n_genes) float

# 标签整数编码
labels_raw  <- scrna_obj$cell_type
cell_types  <- sort(unique(labels_raw))   # 固定排序，保证一致性
label_map   <- setNames(seq_along(cell_types) - 1L, cell_types)
flex_labels <- as.integer(label_map[labels_raw])

cat("scRNA 细胞数:", nrow(flex_expr), "\n")
cat("细胞类型数 :", length(cell_types), "\n")
cat("细胞类型   :", paste(cell_types, collapse = ", "), "\n")

In [ ]:
# ============================================================
# Cell 3: 加载转录本 + 构建 spot 图（核心数据预处理）
# ============================================================
from utils_spot import (
    load_xenium_transcripts,
    bin_transcripts_to_spots,
    unified_normalize_spot,
    build_spot_graph,
)

gene_list = list(xenium_panel_genes)   # 来自 R 的 rpy2 变量
print(f'共用基因数: {len(gene_list)}')

# 缓存文件名（含参数，便于不同配置对比）
cache_tag   = (f"kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
               f"_bin{PARAMS['bin_size']}_val{PARAMS['val_ratio']}")
GRAPH_FILE  = os.path.join(PATHS['cache']['graph'], f'graph_{cache_tag}.pt')
SCALER_FILE = os.path.join(PATHS['cache']['graph'], f'scaler_{cache_tag}.pkl')
COORDS_FILE = os.path.join(PATHS['cache']['graph'], f'spot_coords_{cache_tag}.npy')

CACHE_OK    = all(os.path.exists(p)
                  for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE])
FORCE_REBUILD = False   # 改为 True 可强制重建

if CACHE_OK and not FORCE_REBUILD:
    print('[Cache HIT] 加载图...')
    ckpt         = torch.load(GRAPH_FILE, weights_only=False)
    data         = ckpt['data']
    class_weights = ckpt['class_weights']
    split_info   = ckpt['split_info']
    with open(SCALER_FILE, 'rb') as f:
        fitted_scaler = pickle.load(f)
    spot_coords = np.load(COORDS_FILE)

else:
    print('[Cache MISS] 从转录本构建图...')

    # Step 1: 加载并过滤转录本
    # 过滤流程：Q≥20 → 去控制探针 → 基因对齐
    df_tx = load_xenium_transcripts(
        PATHS['raw']['transcripts'],
        gene_list    = gene_list,
        qv_threshold = PARAMS['qv_threshold'],
    )

    # Step 2: bin 到 spot（5μm）
    spot_expr, spot_coords = bin_transcripts_to_spots(
        df_tx,
        gene_list       = gene_list,
        bin_size        = PARAMS['bin_size'],
        min_transcripts = PARAMS['min_transcripts'],
    )

    # 节点数量检查（超过 50 万建议增大 bin_size）
    n_spots = spot_expr.shape[0]
    print(f'\nSpot 节点数: {n_spots:,}')
    if n_spots > 500_000:
        print(f'⚠️  节点数较多（>{500_000:,}），建议将 bin_size 从 '
              f'{PARAMS["bin_size"]} 增大到 10，然后 FORCE_REBUILD=True 重跑')

    # Step 3: 统一归一化（scaler 只在 scRNA 上 fit）
    print('归一化...')
    scrna_norm, spot_norm, fitted_scaler = unified_normalize_spot(
        np.array(flex_expr, dtype=np.float32),
        spot_expr,
    )

    # Step 4: 构建联合图
    data, class_weights, split_info = build_spot_graph(
        scrna_norm   = scrna_norm,
        spot_norm    = spot_norm,
        spot_coords  = spot_coords,
        scrna_labels = np.array(flex_labels, dtype=np.int64),
        k_feat       = PARAMS['k_feat'],
        k_spatial    = PARAMS['k_spatial'],
        val_ratio    = PARAMS['val_ratio'],
    )

    # 保存缓存
    torch.save({'data': data, 'class_weights': class_weights,
                'split_info': split_info}, GRAPH_FILE)
    with open(SCALER_FILE, 'wb') as f:
        pickle.dump(fitted_scaler, f)
    np.save(COORDS_FILE, spot_coords)
    print(f'已缓存 → {GRAPH_FILE}')

cell_types_list = list(cell_types)   # 供后续 Cell 使用
n_classes = len(cell_types_list)

print(f'\n节点数    : {data.x.shape[0]:,}')
print(f'  scRNA   : {split_info["n_scrna"]:,}')
print(f'  spot    : {split_info["n_spots"]:,}')
print(f'边数      : {data.edge_index.shape[1]:,}')
print(f'特征维度  : {data.x.shape[1]}')
print(f'训练 / 验证: {data.train_mask.sum()} / {data.val_mask.sum()}')
print(f'细胞类型数: {n_classes}')
print('预处理完成 ✓')

In [ ]:
# ============================================================
# Cell 4a: 初始化三种 GNN 模型
# 与原版 train_copy.ipynb 完全相同，一行不改
# ============================================================
from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
from utils import run_experiment

in_dim  = data.x.shape[1]
device  = torch.device(PARAMS['device'])

model_configs = {
    'GCN': GCN_AMP(
        in_channels  = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels = n_classes,
        proj_dim     = PARAMS['proj_dim'],
        dropout      = PARAMS['dropout'],
    ),
    'GraphSAGE': GraphSAGE_AMP(
        in_channels  = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels = n_classes,
        proj_dim     = PARAMS['proj_dim'],
        dropout      = PARAMS['dropout'],
    ),
    'GAT': GAT_AMP(
        in_channels  = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels = n_classes,
        proj_dim     = PARAMS['proj_dim'],
        dropout      = PARAMS['dropout'],
    ),
}
print('模型参数量:')
for name, model in model_configs.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  {name}: {n_params:,}')

In [ ]:
# ============================================================
# Cell 4b: 训练三种 GNN（与原版完全相同）
# ============================================================
gnn_results = {}

for model_name, model in model_configs.items():
    print(f'\n{'='*55}')
    print(f'  训练 {model_name}')
    print(f'{'='*55}')

    result = run_experiment(
        model        = model,
        data         = data,
        class_weights = class_weights,
        n_classes    = n_classes,
        device       = device,
        params       = PARAMS,
        model_name   = model_name,
        save_dir     = PATHS['output']['models'],
    )
    gnn_results[model_name] = result
    print(f'  Best Val F1: {result["best_val_f1"]:.4f}')

print('\n所有模型训练完成 ✓')

In [ ]:
# ============================================================
# Cell 5: 评估 — scRNA 验证集（有 ground truth）
# ============================================================
from eval import compute_metrics
import torch.nn.functional as F

val_idx  = split_info['val_idx']
val_true = np.array(flex_labels, dtype=np.int64)[val_idx]
n_scrna  = split_info['n_scrna']

val_preds     = {}
spot_probas   = {}   # 保存每个模型对 spot 的预测概率

for name, res in gnn_results.items():
    model_eval = res['model'].to('cpu').eval()
    data_cpu   = data.cpu()
    with torch.no_grad():
        log_probs = model_eval(data_cpu)          # (n_total, n_classes)
        proba_all = torch.softmax(log_probs, dim=1).numpy()

    val_preds[name]   = proba_all[val_idx].argmax(axis=1)
    spot_probas[name] = proba_all[n_scrna:]       # spot 节点的概率

print('=' * 65)
print('  scRNA 验证集评估（Ground Truth = 真实细胞类型标签）')
print('=' * 65)
print(f'{"Method":<14} {"Acc":>7} {"F1-Mac":>8} {"F1-Wei":>8} {"Kappa":>8}')
print('-' * 65)
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=n_classes)
    print(f'{method:<14} {m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} '
          f'{m["f1_weighted"]:>8.4f} {m["kappa"]:>8.4f}')
print('=' * 65)

In [ ]:
# ============================================================
# Cell 6: 保存 spot 级别预测结果（含空间坐标，供可视化）
# ============================================================
best_name = max(gnn_results,
                key=lambda n: gnn_results[n]['best_val_f1'])
print(f'最优模型: {best_name} (Val F1={gnn_results[best_name]["best_val_f1"]:.4f})')

best_proba     = spot_probas[best_name]    # (n_spots, n_classes)
best_pred_idx  = best_proba.argmax(axis=1)
best_pred_label= [cell_types_list[i] for i in best_pred_idx]
best_confidence= best_proba.max(axis=1)

# 保存为 CSV（x/y 坐标 + 预测类型 + 置信度）
spot_df = pd.DataFrame({
    'x'          : spot_coords[:, 0],
    'y'          : spot_coords[:, 1],
    'pred_idx'   : best_pred_idx,
    'pred_label' : best_pred_label,
    'confidence' : best_confidence,
})
out_csv = f'{PATHS["output"]["predictions"]}/spot_predictions_{best_name}.csv'
spot_df.to_csv(out_csv, index=False)
print(f'Spot 预测已保存: {out_csv}  ({len(spot_df):,} spots)')

# 所有模型结果保存
for name, proba in spot_probas.items():
    df_all = pd.DataFrame(proba,
        columns=[f'prob_{c}' for c in cell_types_list])
    df_all['x'] = spot_coords[:, 0]
    df_all['y'] = spot_coords[:, 1]
    df_all['pred_label'] = [cell_types_list[i]
                            for i in proba.argmax(axis=1)]
    df_all.to_csv(
        f'{PATHS["output"]["predictions"]}/spot_predictions_{name}_full.csv',
        index=False)
print('所有模型 spot 预测已保存 ✓')

In [ ]:
# ============================================================
# Cell 7: 空间可视化
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(gnn_results), figsize=(7*len(gnn_results), 7))
if len(gnn_results) == 1:
    axes = [axes]

cmap   = plt.cm.get_cmap('tab20', n_classes)
colors = {i: cmap(i) for i in range(n_classes)}

for ax, (model_name, proba) in zip(axes, spot_probas.items()):
    pred_idx = proba.argmax(axis=1)
    c_vals   = [colors[i] for i in pred_idx]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.5, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(f'{model_name}\nVal F1={gnn_results[model_name]["best_val_f1"]:.3f}',
                 fontsize=12)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')

# 图例
patches = [mpatches.Patch(color=colors[i], label=cell_types_list[i])
           for i in range(n_classes)]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.9),
           loc='upper left', fontsize=8, framealpha=0.8)

plt.suptitle(
    f'GNN Spot-level Cell Type Predictions\n'
    f'Kidney IgAN Xenium | bin={PARAMS["bin_size"]}μm | '
    f'No Cell Segmentation Required',
    fontsize=13, y=1.02
)
plt.tight_layout()
out_fig = f'{PATHS["output"]["plots"]}/spot_spatial_all_models.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'空间图已保存: {out_fig}')

In [ ]:
# ============================================================
# Cell 8: Moran's I 空间自相关（与原版评估对接）
# ============================================================
from topact import TopACT

best_pred_labels_int = spot_probas[best_name].argmax(axis=1)

print(f'计算 Moran\'s I（{best_name}）...')
moran_global = TopACT.morans_i(
    labels      = best_pred_labels_int,
    coords      = spot_coords,
    n_neighbors = 10,
)
print(f"全局 Moran's I = {moran_global['I']:.4f}"
      f" (z={moran_global['z_score']:.2f}, p={moran_global['p_value']:.2e})")

print('\n各细胞类型 Moran\'s I:')
per_class = TopACT.per_class_morans_i(
    labels      = best_pred_labels_int,
    coords      = spot_coords,
    cell_types  = cell_types_list,
    n_neighbors = 10,
)
for ct, vals in sorted(per_class.items(),
                        key=lambda x: -x[1]['I']):
    print(f'  {ct:<12} I={vals["I"]:>7.4f}  p={vals["p_value"]:.2e}')